# WORKING WITH LAMBDA SERVICES

## Libraries and functions

In [ ]:

import os
from io import BytesIO
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from lib.frontend_functions import (
    login,
    upload_file,
    read_file,
    list_bucket,
    delete_file,
    parse_txt_data,
    calculate_sigmoid_batch,
    calculate_cost_logistic,
    calculate_gradient_logistic,
    perform_gradient_descent_logistic,
    perform_prediction_logistic,
    normalize_features,
    train_linear_regression,
    predict_linear_regression,
    train_single_linear_regression
)
from lib.models import (
    UploadFileParams,
    DeleteFileParams,
    LogisticCostParams,
    GradientDescentParams,
    NormalizeFeaturesParams,
    PredictLogisticParams,
    LinearGradientDescentParams,
    PredictLinearParams,
    LinearGradientSingleDescentParams
)

from lib.graph import plot_data, draw_vthresh
from lib.graph import plot_cost_i_w, norm_plot
from lib.graph import DLC

from IPython.display import Image, display

from PIL import Image as PILImage

plt.style.use('./lib/deeplearning.mplstyle')
plt.style.use('seaborn-v0_8')
sns.set_style('darkgrid')


## Start Timer

In [ ]:
def start_timer():
    print("Timer started. Press Enter to stop.")
    start = time.time()
    input()
    end = time.time()
    duration = round(end - start, 2)
    print(f"Elapsed time: {duration} seconds")

In [ ]:
print("Timer started.")
start_process = time.time()
start_time = datetime.now()
start_format = start_time.strftime('%Y-%m-%d %H:%M:%S')
start_format

## Endpoints

In [ ]:
url_token = 'https://api.bearsoft.com.bo'
# url_token = 'http://localhost:3000'
url_files = 'https://api.bearsoft.com.bo'
# url_files = 'http://localhost:3000'
url_ml_functions = 'https://api.bearsoft.com.bo'
# url_ml_functions = 'http://localhost:3000'

## Auth Module

### Get Token

In [ ]:
# token = login(url_token, 'raforios@gmail.com', 'MotoPassword')
token = login(url_token)
token

## Files Module

### Read File from Bucket S3

In [ ]:
data = read_file(token, url_files, 's3/read', 'ml-data-file-handler', 'Sample_Data_for_Plotting_and_Filtering.csv')

data

### Upload file to Bucket S3

In [ ]:
local_path_base = '/Users/rafael/Work/projects/back/SmartBear/data'
file_name_to_upload = 'localization.csv'
bucket_name = 'ml-data-file-handler'
s3_file_path = ''

full_local_file_path_for_upload = os.path.join(local_path_base, file_name_to_upload)

In [ ]:
upload_params = UploadFileParams(
    full_local_file_path = full_local_file_path_for_upload,
    bucket = bucket_name,
    validation = True,
    file_path = s3_file_path
)

In [ ]:
data = upload_file(token, url_files, 's3/upload-presigned', upload_params)
data

In [ ]:
# Reading data from S3 bucket
data = read_file(token, url_files, 's3/read', bucket_name, file_name_to_upload, delimiter = ';')

data


### Upload Binary file to Bucket S3

In [ ]:
FILE_NAME = 'r.png'
LOCAL_FILE_PATH = '/Users/rafael/Downloads/'

In [ ]:
full_local_file_path_for_upload = os.path.join(LOCAL_FILE_PATH, FILE_NAME)

upload_params = UploadFileParams(
    full_local_file_path = full_local_file_path_for_upload,
    bucket = bucket_name,
    validation = True,
    file_path = s3_file_path
)

In [ ]:
# upload_binary = upload_file(token, url_files, 's3/upload-presigned', upload_params)
# upload_binary

### Read Binary file from a Bucket S3

In [ ]:
downloaded_content = read_file(token, url_files, 's3/read', bucket_name, FILE_NAME)

downloaded_content


In [ ]:
try:
    if isinstance(downloaded_content, bytes):
        print('Imagen descargada exitosamente. Redimensionando y mostrando en el notebook...')

        # 2. Abrir la imagen desde los bytes
        img = PILImage.open(BytesIO(downloaded_content))

        # 3. Redimensionar la imagen a un tamaño razonable para la pantalla
        max_size = (1024, 1024) # Puedes ajustar estos valores según tus necesidades
        img.thumbnail(max_size) # Redimensiona la imagen sin perder la proporción

        # 4. Convertir la imagen redimensionada de nuevo a bytes
        # Usamos BytesIO para guardarla en memoria sin tocar el disco
        buffered = BytesIO()
        img.save(buffered, format = 'PNG') # Se recomienda PNG para evitar pérdida de calidad

        # 5. Mostrar la imagen redimensionada con IPython.display
        display(Image(data = buffered.getvalue()))
    else:
        print('El contenido descargado no es un archivo binario. No se puede mostrar como imagen.')

except Exception as e:
    print(f'Ocurrió un error al intentar procesar o mostrar la imagen: {e}')

### List content from a Bucket S3

In [ ]:
bucket_name = 'ml-data-file-handler'

data = list_bucket(token, url_files, 's3/list-files', bucket_name)

data

### Delete a file into a Bucket S3

In [ ]:
delete_params = DeleteFileParams(
    bucket_name,
    file_name = 'A3.csv',
    file_path = ''
)

data = delete_file(token, url_files, 's3/delete', delete_params)

data


In [ ]:
data = list_bucket(token, url_files, 's3/list-files', bucket_name)

data

## Classification Module

### Upload file to Bucket S3 using File Microservice.

In [ ]:
local_path_base = '/Users/rafael/Work/projects/back/SmartBear/data'
file_name_to_upload = 'ex2data1.txt'
bucket_name = 'ml-data-file-handler'
s3_file_path = ''

full_local_file_path_for_upload = os.path.join(local_path_base, file_name_to_upload)

In [ ]:
# file_name = 'DetalleVentas.xlsx'

upload_params = UploadFileParams(
    full_local_file_path = full_local_file_path_for_upload,
    bucket = bucket_name,
    validation = True,
    file_path = s3_file_path
)

In [ ]:
data = upload_file(token, url_files, 's3/upload-presigned', upload_params)
data

### Case 1: Límite y umbral de clasificación

In [ ]:
X = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1]).reshape(-1,1) 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
plot_data(X, y, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$')
ax.set_xlabel('$x_0$')
plt.show()

In [ ]:
# Plot sigmoid(z) over a range of values from -10 to 10
z = np.arange(-10, 11)

fig,ax = plt.subplots(1, 1, figsize = (5, 3))
sigmoid = calculate_sigmoid_batch(token, url_ml_functions, 'classification/sigmoid-batch', z)
# Plot z vs sigmoid(z)
ax.plot(z, sigmoid, c = 'b')

ax.set_title('Sigmoid function')
ax.set_ylabel('sigmoid(z)')
ax.set_xlabel('z')
draw_vthresh(ax, 0)

In [ ]:
# Choose values between 0 and 6
x0 = np.arange(0, 6)

x1 = 3 - x0
fig,ax = plt.subplots(1, 1, figsize = (5, 4))
# Plot the decision boundary
ax.plot(x0,x1, c = 'b')
ax.axis([0, 4, 0, 3.5])

# Fill the region below the line
ax.fill_between(x0, x1, alpha = 0.2)

# Plot the original data
plot_data(X, y, ax)
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')
plt.show()

### Case 2: Gradiente y predicción sin estandarización

In [ ]:
# Reading data from S3 bucket
data = read_file(token, url_files, 's3/read', bucket_name, file_name_to_upload)

data


In [ ]:
X_train, y_train = parse_txt_data(data, delimiter = ',', until_col = 2, skip_rows = 0)

In [ ]:
print(f'First five elements in X_train are: {X_train[:5]}')
print(f'Type of X_train: {type(X_train)}')

In [ ]:
print(f'First five elements in y_train are: {y_train[:5]}')
print(f'Type of y_train: {type(y_train)}')

In [ ]:
print (f'The shape of X_train is: {X_train.shape}')
print (f'The shape of y_train is: {y_train.shape}')
print (f'We have m = {(len(y_train))} training examples')

In [ ]:
pos , neg = (y_train == 1).reshape(100, 1), (y_train == 0).reshape(100, 1)
plt.scatter(X_train[pos[:,0], 0],X_train[pos[:,0], 1], marker = '*')
plt.scatter(X_train[neg[:,0], 0],X_train[neg[:,0], 1], c = 'r', marker = 'o', s = 10)
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Admitted','Not admitted'], loc = 0)

#### Sigmoid function

In [ ]:
# Note: You can edit this value
value = 0
# Sigmoid with a numeric value
sigmoid = calculate_sigmoid_batch(token, url_ml_functions, 'classification/sigmoid-batch', np.array([value]))

print (f'sigmoid({value}) = {sigmoid}')

In [ ]:
sigmoid = calculate_sigmoid_batch(token, url_ml_functions, 'classification/sigmoid-batch', np.array([-1, 0, 1, 2]))

print (f'sigmoid([ -1, 0, 1, 2]) =  {str(sigmoid)}')

#### Cost function for logistic regression

In [ ]:
# Compute and display cost with w and b initialized to zeros
m, n = X_train.shape
cost_params = LogisticCostParams(
    x_matrix = X_train,
    y = y_train,
    w = np.zeros(n),
    b = 0.
)
cost = calculate_cost_logistic(token, url_ml_functions, 'classification/cost-logistic', cost_params)

print(f'Cost at initial w and b (zeros): {cost: .3f}')

In [ ]:
# Compute and display cost with non-zero w and b
cost_params = LogisticCostParams(
    x_matrix = X_train,
    y = y_train,
    w = np.array([0.2, 0.2]),
    b = -24.
)
cost = calculate_cost_logistic(token, url_ml_functions, 'classification/cost-logistic', cost_params)

print(f'Cost at test w and b (non-zeros): {cost: .3f}')



#### Gradient for logistic regression

In [ ]:
# Compute and display gradient with w and b initialized to zeros
gradient_params = LogisticCostParams(
    x_matrix = X_train,
    y = y_train,
    w = np.zeros(n),
    b = 0.
)

gradient_response = calculate_gradient_logistic(token, url_ml_functions, 'classification/gradient-logistic', gradient_params)
dj_db = gradient_response.get('dj_db')
dj_dw = gradient_response.get('dj_dw')

print(f'dj_db at initial w and b (zeros): {dj_db: .3f}' )
print(f'dj_dw at initial w and b (zeros): {np.array(dj_dw)}' )

In [ ]:
# Compute and display cost and gradient with non-zero w and b
gradient_params = LogisticCostParams(
    x_matrix = X_train,
    y = y_train,
    w = np.array([ 0.2, -0.5]),
    b = -24
)

gradient_response = calculate_gradient_logistic(token, url_ml_functions, 'classification/gradient-logistic', gradient_params)
dj_db = gradient_response.get('dj_db')
dj_dw = gradient_response.get('dj_dw')

print(f'dj_db at initial w and b (zeros): {dj_db: .3f}' )
print(f'dj_dw at initial w and b (zeros): {np.array(dj_dw)}' )


#### Learning parameters using gradient descent 

In [ ]:
np.random.seed(1)
initial_w = 0.01 * (np.random.rand(2) - 0.5)
initial_b = -8

# Some gradient descent settings
iterations = 3000
alpha = 0.001

gd_params = GradientDescentParams(
    x_matrix = X_train,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = alpha,
    num_iters = iterations
)

gd_results = perform_gradient_descent_logistic(token, url_ml_functions, 'classification/train-logistic-regression', gd_params)

w = np.array(gd_results['w'])
b = gd_results['b']
J_history = gd_results['J_history']
p_history = gd_results['p_history']

print(f"Final w: {w}")
print(f"Final b: {b}")
print(f"Costos en la última iteración: {J_history[-1]}")


In [ ]:
plt.plot(J_history)
plt.xlabel('Iteration')
plt.ylabel('J')
plt.title('Cost function using Gradient Descent')

In [ ]:
plt.scatter(X_train[pos[:,0],0],X_train[pos[:,0],1], marker = '*')
plt.scatter(X_train[neg[:,0],0],X_train[neg[:,0],1], c = 'r', marker = 'o', s = 10)

x_value = np.array([np.min(X_train[:,1]), np.max(X_train[:,1])])
y_value = -(b + w[0] * x_value)/w[1]
plt.plot(x_value,y_value, '#FF9300')
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Admitted','Not admitted'], loc = 0)

#### Predictions

In [ ]:
# Test your predict code
np.random.seed(1)
tmp_w = np.random.randn(2)
tmp_b = 0.3    
tmp_X = np.random.randn(4, 2) - 0.5

prediction_params = PredictLogisticParams(
    x_matrix = tmp_X,
    w = tmp_w,
    b = tmp_b
)

predictions = perform_prediction_logistic(token, url_ml_functions, 'classification/predict-logistic-classification', prediction_params)

print(f'Output of predict: {predictions}')

In [ ]:
#Compute accuracy on our training set
prediction_params = PredictLogisticParams(
    x_matrix = X_train,
    w = w,
    b = b
)

predictions_response = perform_prediction_logistic(token, url_ml_functions, 'classification/predict-logistic-classification', prediction_params)
p_from_api = np.array(predictions_response.get('predictions'))

print(f'Train Accuracy: {(np.mean(p_from_api == y_train) * 100): .2f}%')

In [ ]:
x_test = np.array([55, 85])
# x_test = (x_test - X_mean)/X_std
sig = np.dot(x_test, w) + b
# Sigmoid with a numeric value
prob = calculate_sigmoid_batch(token, url_ml_functions, 'classification/sigmoid-batch', [sig])

print(f'For a student with scores {x_test[0]} and {x_test[1]}, we predict an admission probability of: {(prob[0] * 100):.2f}%')

In [ ]:
plt.scatter(x_test[0], x_test[1], c = '#FF40FF', marker='v', s = 80, label='(45, 85)')
plt.scatter(X_train[pos[:,0],0],X_train[pos[:,0],1], marker = '*')
plt.scatter(X_train[neg[:,0],0],X_train[neg[:,0],1], c = 'r', marker = 'o', s = 10)

x_value = np.array([np.min(X_train[:,1]), np.max(X_train[:,1])])
y_value = -(b + w[0] * x_value)/w[1]
plt.plot(x_value,y_value, '#FF9300')
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Predict', 'Admitted','Not admitted'],loc = 0)

### Case 3: Gradiente y predicción con estandarización

In [ ]:
normalization_params = NormalizeFeaturesParams(
    x_matrix = X_train
)

normalized_response = normalize_features(token, url_ml_functions,'common/normalize-features', normalization_params)


print("\nRespuesta de normalización recibida:")
print(f"X Normalizada (x_norm): {normalized_response.get('x_norm')}")
print(f"Media (mu): {normalized_response.get('mu')}")
print(f"Desviación Estándar (sigma): {normalized_response.get('sigma')}")

# Puedes convertir las listas a arrays de NumPy si las necesitas así para cálculos posteriores
X_train_normalized_api = np.array(normalized_response.get('x_norm'))
X_mean_api = np.array(normalized_response.get('mu'))
X_std_api = np.array(normalized_response.get('sigma'))

print(f"\nShape de X normalizada desde API: {X_train_normalized_api.shape}")
print(f"Tipo de X normalizada desde API: {type(X_train_normalized_api)}")


In [ ]:
print('First five elements in X_train_normalized are:\n', X_train_normalized_api[:5])
print('Type of X_train_normalized:',type(X_train_normalized_api))

In [ ]:
print(f'First five elements in y_train are: {y_train[:5]}')
print(f'Type of y_train: {type(y_train)}')

In [ ]:
print (f'The shape of X_train_normalized is: {X_train_normalized_api.shape}')
print (f'The shape of y_train is: {y_train.shape}')
print (f'We have m = {(len(y_train))} training examples')

In [ ]:
pos , neg = (y_train == 1).reshape(100, 1), (y_train == 0).reshape(100, 1)
plt.scatter(X_train_normalized_api[pos[:,0], 0],X_train_normalized_api[pos[:,0], 1], marker = '*')
plt.scatter(X_train_normalized_api[neg[:,0], 0],X_train_normalized_api[neg[:,0], 1], c = 'r', marker = 'o', s = 10)
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Admitted','Not admitted'], loc = 0)

#### Cost function for logistic regression

In [ ]:
m, n = X_train_normalized_api.shape

# Compute and display cost with w and b initialized to zeros
cost_params = LogisticCostParams(
    x_matrix = X_train_normalized_api,
    y = y_train,
    w = np.zeros(n),
    b = 0.
)

cost = calculate_cost_logistic(token, url_ml_functions, 'classification/cost-logistic', cost_params)

print(f'Cost at initial w and b (zeros): {cost :.3f}')

#### Gradient for logistic regression

In [ ]:
# Compute and display gradient with w and b initialized to zeros
gradient_params = LogisticCostParams(
    x_matrix = X_train_normalized_api,
    y = y_train,
    w = np.array([ 0.2, -0.5]),
    b = -24
)

gradient_response = calculate_gradient_logistic(token, url_ml_functions, 'classification/gradient-logistic', gradient_params)
dj_db = gradient_response.get('dj_db')
dj_dw = gradient_response.get('dj_dw')

print(f'dj_db at initial w and b (zeros): {dj_db: .3f}' )
print(f'dj_dw at initial w and b (zeros): {np.array(dj_dw)}' )


In [ ]:
# Compute and display cost and gradient with non-zero w and b
gradient_params = LogisticCostParams(
    x_matrix = X_train_normalized_api,
    y = y_train,
    w = np.zeros(n),
    b = 0.
)

gradient_response = calculate_gradient_logistic(token, url_ml_functions, 'classification/gradient-logistic', gradient_params)
dj_db = gradient_response.get('dj_db')
dj_dw = gradient_response.get('dj_dw')

print(f'dj_db at initial w and b (zeros): {dj_db: .3f}' )
print(f'dj_dw at initial w and b (zeros): {np.array(dj_dw)}' )


#### Learning parameters using gradient descent 

In [ ]:
np.random.seed(1)
initial_w = 0.01 * (np.random.rand(2) - 0.5)
initial_b = -8

# Some gradient descent settings
iterations = 2000
alpha = 0.01

gd_params = GradientDescentParams(
    x_matrix = X_train_normalized_api,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = alpha,
    num_iters = iterations
)

gd_results = perform_gradient_descent_logistic(token, url_ml_functions, 'classification/train-logistic-regression', gd_params)

w = np.array(gd_results['w'])
b = gd_results['b']
J_history = gd_results['J_history']
p_history = gd_results['p_history']

print(f"Final w: {w}")
print(f"Final b: {b}")
print(f"Costos en la última iteración: {J_history[-1]}")


In [ ]:
# Some gradient descent settings
iterations = 3500
alpha = 0.9

gd_params = GradientDescentParams(
    x_matrix = X_train_normalized_api,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = alpha,
    num_iters = iterations
)

gd_results = perform_gradient_descent_logistic(token, url_ml_functions, 'classification/train-logistic-regression', gd_params)

w = np.array(gd_results['w'])
b = gd_results['b']
J_history = gd_results['J_history']
p_history = gd_results['p_history']

print(f"Final w: {w}")
print(f"Final b: {b}")
print(f"Costos en la última iteración: {J_history[-1]}")


In [ ]:
plt.plot(J_history)
plt.xlabel('Iteration')
plt.ylabel('J')
plt.title('Cost function using Gradient Descent')

In [ ]:
plt.scatter(X_train_normalized_api[pos[:,0],0],X_train_normalized_api[pos[:,0], 1], marker = '*')
plt.scatter(X_train_normalized_api[neg[:,0],0],X_train_normalized_api[neg[:,0], 1], c = 'r', marker = 'o', s = 10)

x_value = np.array([np.min(X_train_normalized_api[:, 1]), np.max(X_train_normalized_api[:, 1])])
y_value = -(b + w[0] * x_value)/w[1]
plt.plot(x_value,y_value, '#FF9300')
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Admitted','Not admitted'], loc = 0)

#### Predictions

In [ ]:
#Compute accuracy on our training set
prediction_params = PredictLogisticParams(
    x_matrix = X_train_normalized_api,
    w = w,
    b = b
)

predictions_response = perform_prediction_logistic(token, url_ml_functions, 'classification/predict-logistic-classification', prediction_params)
p_from_api = np.array(predictions_response.get('predictions'))

print(f'Train Accuracy: {(np.mean(p_from_api == y_train) * 100): .2f}%')


In [ ]:
x_test = np.array([55, 85])
x_test = (x_test - X_mean_api)/X_std_api
sig = np.dot(x_test, w) + b
# Sigmoid with a numeric value
prob = calculate_sigmoid_batch(token, url_ml_functions, 'classification/sigmoid-batch', [sig])

print(f'For a student with scores {x_test[0]: .2f} and {x_test[1]: .2f}, we predict an admission probability of: {(prob[0] * 100):.2f}%')

In [ ]:
plt.scatter(x_test[0], x_test[1], c = '#FF40FF', marker='v', s = 80, label='(45, 85)')
plt.scatter(X_train_normalized_api[pos[:,0],0],X_train_normalized_api[pos[:,0],1], marker = '*')
plt.scatter(X_train_normalized_api[neg[:,0],0],X_train_normalized_api[neg[:,0],1], c = 'r', marker = 'o', s = 10)

x_value = np.array([np.min(X_train_normalized_api[:,1]), np.max(X_train_normalized_api[:,1])])
y_value = -(b + w[0] * x_value)/w[1]
plt.plot(x_value,y_value, '#FF9300')
plt.xlabel('Exam 1 score')
plt.ylabel('Exam 2 score')
plt.legend(['Predict', 'Admitted','Not admitted'],loc = 0)


## Predictions Module

### Upload file to Bucket S3 using File Microservice.

In [ ]:
local_path_base = '/Users/rafael/Work/projects/back/SmartBear/data'
file_name_to_upload = 'houses.txt'
bucket_name = 'ml-data-file-handler'
s3_file_path = ''

full_local_file_path_for_upload = os.path.join(local_path_base, file_name_to_upload)

In [ ]:
upload_params = UploadFileParams(
    full_local_file_path = full_local_file_path_for_upload,
    bucket = bucket_name,
    validation = True,
    file_path = s3_file_path
)

### Case 1: Multi linear regression

In [ ]:
data = upload_file(token, url_files, 's3/upload-presigned', upload_params)
data

In [ ]:
# Reading data from S3 bucket
data = read_file(token, url_files, 's3/read', bucket_name, file_name_to_upload)

data

In [ ]:
# load the dataset
X_train, y_train = parse_txt_data(data, delimiter = ',', until_col = 4, skip_rows = 1)
X_features = ['size(sqft)','bedrooms','floors','age']

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (12, 3), sharey = True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:, i], y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

#### $\alpha$ = 9.9e-7

In [ ]:
_, n = X_train.shape
initial_w = np.zeros(n)
initial_b = 0

params_train = LinearGradientDescentParams(
    x = X_train,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = 9.9e-7,
    num_iters = 1000
)

response = train_linear_regression(token, url_ml_functions, 'prediction/train-linear-regression', params = params_train)

hist = {
    'iter': list(range(1, response.get('num_iters') + 1)),
    'cost': response.get('J_history'),
    'params': response.get('p_history')
}

# Obtener los valores finales
final_cost = response['J_history'][-1]
final_w = response['w_final']
final_b = response['b_final']

# print(f"Final Iteration: {final_iteration}")
print("---------------------------------------------------")
print(f"Cost:   {final_cost:.2e}")
print(f"w:      {final_w}")
print(f"b:      {final_b:.2e}")
print("---------------------------------------------------")

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

#### $\alpha$ = 9e-7

In [ ]:
params_train = LinearGradientDescentParams(
    x = X_train,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = 9e-7,
    num_iters = 1000
)

response = train_linear_regression(token, url_ml_functions, 'prediction/train-linear-regression', params = params_train)

hist = {
    'iter': list(range(1, response.get('num_iters') + 1)),
    'cost': response.get('J_history'),
    'params': response.get('p_history')
}

# Obtener los valores finales
final_cost = response['J_history'][-1]
final_w = response['w_final']
final_b = response['b_final']

# print(f"Final Iteration: {final_iteration}")
print("---------------------------------------------------")
print(f"Cost:   {final_cost:.2e}")
print(f"w:      {final_w}")
print(f"b:      {final_b:.2e}")
print("---------------------------------------------------")

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

#### $\alpha$ = 1e-7

In [ ]:
params_train = LinearGradientDescentParams(
    x = X_train,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = 1e-7,
    num_iters = 1000
)

response = train_linear_regression(token, url_ml_functions, 'prediction/train-linear-regression', params = params_train)

hist = {
    'iter': list(range(1, response.get('num_iters') + 1)),
    'cost': response.get('J_history'),
    'params': response.get('p_history')
}

# Obtener los valores finales
final_cost = response['J_history'][-1]
final_w = response['w_final']
final_b = response['b_final']

# print(f"Final Iteration: {final_iteration}")
print("---------------------------------------------------")
print(f"Cost:   {final_cost:.2e}")
print(f"w:      {final_w}")
print(f"b:      {final_b:.2e}")
print("---------------------------------------------------")

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

#### Standarization

In [ ]:
mu     = np.mean(X_train, axis = 0)
sigma  = np.std(X_train, axis = 0)
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma 

fig,ax = plt.subplots(1, 3, figsize = (12, 3))
ax[0].scatter(X_train[:, 0], X_train[:, 3])
ax[0].set_xlabel(X_features[0])
ax[0].set_ylabel(X_features[3])
ax[0].set_title('unnormalized')
ax[0].axis('equal')

ax[1].scatter(X_mean[:, 0], X_mean[:, 3])
ax[1].set_xlabel(X_features[0])
ax[0].set_ylabel(X_features[3])
ax[1].set_title(r'X - $\mu$')
ax[1].axis('equal')

ax[2].scatter(X_norm[:, 0], X_norm[:, 3])
ax[2].set_xlabel(X_features[0])
ax[0].set_ylabel(X_features[3])
ax[2].set_title(r'Z-score normalized')
ax[2].axis('equal')
plt.tight_layout(rect = [0, 0.03, 1, 0.95])
fig.suptitle('distribution of features before, during, after normalization')
plt.show()

In [ ]:
# normalize the original features
normalization_params = NormalizeFeaturesParams(
    x_matrix = X_train
)

normalized_response = normalize_features(token, url_ml_functions,'common/normalize-features', normalization_params)


print("\nRespuesta de normalización recibida:")
print(f"X Normalizada (x_norm): {normalized_response.get('x_norm')}")
print(f"Media (mu): {normalized_response.get('mu')}")
print(f"Desviación Estándar (sigma): {normalized_response.get('sigma')}")

# Puedes convertir las listas a arrays de NumPy si las necesitas así para cálculos posteriores
X_train_normalized_api = np.array(normalized_response.get('x_norm'))
X_mean_api = np.array(normalized_response.get('mu'))
X_std_api = np.array(normalized_response.get('sigma'))

print(f"\nShape de X normalizada desde API: {X_train_normalized_api.shape}")
print(f"Tipo de X normalizada desde API: {type(X_train_normalized_api)}")


In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (12, 3))
for i in range(len(ax)):
    norm_plot(ax[i], X_train[:, i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel('count')
fig.suptitle('distribution of features before normalization')
plt.show()
fig, ax = plt.subplots(1, 4, figsize = (12, 3))
for i in range(len(ax)):
    norm_plot(ax[i], X_train_normalized_api[:, i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel('count'); 
fig.suptitle('distribution of features after normalization')

plt.show()

#### With Normalization and $\alpha$ = 3.0e-1

In [ ]:
params_train = LinearGradientDescentParams(
    x = X_train_normalized_api,
    y = y_train,
    w_in = initial_w,
    b_in = initial_b,
    alpha = 3.0e-1,
    num_iters = 1000
)

response = train_linear_regression(token, url_ml_functions, 'prediction/train-linear-regression', params = params_train)

hist = {
    'iter': list(range(1, response.get('num_iters') + 1)),
    'cost': response.get('J_history'),
    'params': response.get('p_history')
}

# Obtener los valores finales
final_cost = response['J_history'][-1]
final_w = response['w_final']
final_b = response['b_final']

# print(f"Final Iteration: {final_iteration}")
print("---------------------------------------------------")
print(f"Cost:   {final_cost:.2e}")
print(f"w:      {final_w}")
print(f"b:      {final_b:.2e}")
print("---------------------------------------------------")

In [ ]:
#predict target using normalized features
m = X_train_normalized_api.shape[0]
params_predict = PredictLinearParams(
    x_test = X_train_normalized_api,
    w = final_w,
    b = final_b
)

yp = predict_linear_regression(token, url_ml_functions, 'prediction/predict-linear-regression', params = params_predict)


In [ ]:
# plot predictions and targets versus original features    
fig, ax = plt.subplots(1,4,figsize = (12, 3), sharey = True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:, i], y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:, i], yp, color = DLC['dlorange'], label = 'predict')
ax[0].set_ylabel('Price')
ax[0].legend()
fig.suptitle('target versus prediction using z-score normalized model')
plt.show()

In [ ]:
yp[:4]

In [ ]:
y_train[:4]

In [ ]:
X_mean_api

In [ ]:
X_std_api

In [ ]:
# First, normalize out example.
x_house = np.array([500, 3, 3, 20])
x_house_norm = (x_house - X_mean_api) / X_std_api
x_house_norm
params_predict = PredictLinearParams(
    x_test = [x_house_norm.tolist()],
    w = final_w,
    b = final_b
)

x_house_predict_response = predict_linear_regression(token, url_ml_functions, 'prediction/predict-linear-regression', params = params_predict)

x_house_predict = x_house_predict_response[0]

print(f' predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict * 1000: 0.0f}')


#### Temperature Prediction

In [ ]:
# Upload file to Bucket S3 using File Microservice.
local_path_base = '/Users/rafael/Work/projects/back/SmartBear/data'
file_name_to_upload = 'celsius_a_fahrenheit.csv'
bucket_name = 'ml-data-file-handler'
s3_file_path = ''

full_local_file_path_for_upload = os.path.join(local_path_base, file_name_to_upload)

upload_params = UploadFileParams(
    full_local_file_path = full_local_file_path_for_upload,
    bucket = bucket_name,
    validation = True,
    file_path = s3_file_path
)

data = upload_file(token, url_files, 's3/upload-presigned', upload_params)
data

In [ ]:
# Reading data from S3 bucket
temperature_df = read_file(token, url_files, 's3/read', bucket_name, file_name_to_upload)

temperature_df

In [ ]:
# Plot the data points
plt.scatter(x = temperature_df['Celsius'], y = temperature_df['Fahrenheit'], marker='o', c='b')
# Set the title
plt.title('TEMPERATURES')
# Set the y-axis label
plt.ylabel('Fahrenheit')
# Set the x-axis label
plt.xlabel('Celsius')
plt.show()

In [ ]:
training_data = temperature_df.sample(frac = 0.8, random_state = 0)
training_label = training_data.pop('Fahrenheit')

In [ ]:
# initialize parameters
params_train = LinearGradientSingleDescentParams(
    x = training_data['Celsius'].to_numpy(),
    y = training_label.to_numpy(),
    w_in = 0,
    b_in = 0,
    alpha = 3.0e-3,
    num_iters = 3100
)

# run gradient descent
response = train_single_linear_regression(token, url_ml_functions, 'prediction/train-linear-regression', params = params_train)

hist = {
    'iter': list(range(1, response.get('num_iters') + 1)),
    'cost': response.get('J_history'),
    'params': response.get('p_history')
}

# Obtener los valores finales
final_cost = response['J_history'][-1]
final_w = response['w_final']
final_b = response['b_final']

# print(f"Final Iteration: {final_iteration}")
print("---------------------------------------------------")
print(f"Cost:   {final_cost:.2e}")
print(f"w:      {final_w}")
print(f"b:      {final_b:.2e}")
print("---------------------------------------------------")

In [ ]:
print(f'w: {final_w:0.2f} ---- b: {final_b:0.2f}')
print(f'25 °C prediction: {final_w * 25 + final_b: 0.1f} °F')
print(f'30  °C prediction: {final_w * 30 + final_b: 0.1f} °F')
print(f'15  °C prediction: {final_w * 15 + final_b: 0.1f} °F')

In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12,4))
ax1.plot(response['J_history'][:100])
ax2.plot(1000 + np.arange(len(response['J_history'][1000:])), response['J_history'][1000:])
ax1.set_title('Cost vs. iteration(start)');  ax2.set_title('Cost vs. iteration (end)')
ax1.set_ylabel('Cost')            ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')  ;  ax2.set_xlabel('iteration step') 
plt.show()


## End Timer

In [ ]:
end_process = time.time()
duration = round(end_process - start_process, 2)
end_time = datetime.now()
elapsed_time = end_time - start_time
end_format = end_time.strftime('%Y-%m-%d %H:%M:%S')

end_format

In [ ]:
print(f"Elapsed time: {duration} seconds")
elapsed_time

In [ ]:
# 1. Cargar el archivo CSV
# Al estar separado por ",", no necesitas especificar el separador (es el default)
# df = pd.read_csv('stock.csv')

In [ ]:
# 2. (Opcional) Ver las primeras filas para confirmar que cargó bien
# df.head()

In [ ]:
# Reemplaza 'codigo' y 'cantidad' por los nombres exactos de tus columnas
# df_agrupado = df.groupby(['code', 'description', 'unit'])['stock'].sum().reset_index()

# 3. Guardar el resultado en Excel
# df_agrupado.to_excel('stock.xlsx', index=False)

print("¡Proceso completado! Las filas duplicadas se han sumado.")

In [ ]:
# 3. Exportar a Excel
# Usamos index=False para que no guarde la columna de números de fila
# df.to_excel('stock.xlsx', index=False)

# print("¡Archivo exportado con éxito!")